# US Retail Sector — Credit Deep Dive

---

## What You Have

Four datasets on 25 US retail companies (investment grade through high yield):

| Dataset | What's in it |
|---------|-------------|
| **Company financials** | 12 quarters of revenue, EBITDA, leverage, coverage, margins, FCF |
| **Bond spreads** | Daily mid/bid/ask OAS and bid-ask spread for each company's benchmark bond |
| **Bond reference** | Coupon, maturity, amount outstanding, liquidity tier |
| **Credit events** | Rating changes, earnings surprises, covenant issues |

## What To Do

**Step 1 (~1 hour): Exploratory Analysis**

We want to understand the relationship between the companies' fundamental positions and their spread (price). Do some exploratory analysis and produce charts or tables that help us understand:

- What variables are correlated with spread? What isn't?
- Which companies are improving or deteriorating based on the fundamentals?
- Are there data quality issues we should be aware of?

Use any AI tools you want — for writing functions, getting ideas for analysis, debugging, etc. We'll discuss your observations when you're done.

**Break: Discussion**

We'll talk through what you found and decide together what model to build.

**Step 2 (~1 hour): Build a Simple Model**

Based on our discussion, build a quantitative model. Two natural options:

- **Cross-sectional regression** — model log(OAS) as a function of fundamentals (e.g. leverage, coverage). Plot model OAS vs. actual OAS to see who is rich or cheap in the sector.
- **Z-score relative value** — compute each bond's z-score relative to its own spread history. Who is wide or tight vs. themselves?

Produce a summary table or chart of your findings. We'll discuss what trades the model suggests — what to go long, what to go short — and the limitations of the approach.

---

## Setup

Run the two cells below to load data and set up helper functions. Then start your analysis in the cells that follow.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

BASE_URL = "https://raw.githubusercontent.com/johnnyp2587/assessment_data/main"

fin     = pd.read_csv(f"{BASE_URL}/quarterly_financials.csv")
spreads = pd.read_csv(f"{BASE_URL}/daily_spreads.csv")
bonds   = pd.read_csv(f"{BASE_URL}/bond_reference.csv")
events  = pd.read_csv(f"{BASE_URL}/credit_events.csv")

spreads["date"] = pd.to_datetime(spreads["date"])

print(f"Financials : {fin.shape[0]} rows — {fin['ticker'].nunique()} companies, {fin['quarter'].nunique()} quarters")
print(f"Spreads    : {spreads.shape[0]} rows — {spreads['ticker'].nunique()} tickers, {spreads['date'].nunique()} trading days")
print(f"Bonds      : {bonds.shape[0]} records")
print(f"Events     : {events.shape[0]} events")
print(f"\nQuarters: {fin['quarter'].min()} to {fin['quarter'].max()}")
print(f"Spread dates: {spreads['date'].min().date()} to {spreads['date'].max().date()}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# HELPER FUNCTIONS — use these to save time on data wrangling
# ═══════════════════════════════════════════════════════════════

def latest_snapshot():
    """
    Returns a single DataFrame with one row per company:
    latest quarter financials + latest spread + bond reference data.

    Columns include: ticker, company_name, sub_sector, revenue_mm, ebitda_mm,
    ebitda_margin, leverage_ratio, interest_coverage, free_cash_flow_mm,
    mid_oas_bps, bid_ask_bps, coupon, maturity_date, years_to_maturity,
    amount_outstanding_mm, indicative_price, liquidity_tier, initial_rating, etc.
    """
    latest_fin = fin[fin["quarter"] == fin["quarter"].max()].copy()
    latest_sp = spreads[spreads["date"] == spreads["date"].max()][["ticker", "mid_oas_bps", "bid_ask_bps"]]
    df = latest_fin.merge(bonds, on="ticker", suffixes=("", "_bond"))
    df = df.merge(latest_sp, on="ticker")
    return df


def quarterly_with_spreads():
    """
    Returns financials with quarter-end spread attached.
    One row per company per quarter.
    Useful for plotting how financials and spreads move together over time.
    """
    # Get the last spread observation in each quarter
    sp = spreads.copy()
    sp["quarter"] = sp["date"].dt.to_period("Q").astype(str).str.replace("-", "")  # e.g. "2024Q4"
    # Fix format to match fin: "2024Q4" not "2024Q04" etc
    sp["quarter"] = sp["date"].apply(lambda d: f"{d.year}Q{d.quarter}")
    qtr_sp = sp.groupby(["ticker", "quarter"]).agg(
        mid_oas_bps=("mid_oas_bps", "last"),
        bid_ask_bps=("bid_ask_bps", "last"),
    ).reset_index()
    df = fin.merge(qtr_sp, on=["ticker", "quarter"], how="left")
    return df


def spread_history(tickers=None):
    """
    Returns daily spread data for the given tickers (or all if None).
    Columns: date, ticker, mid_oas_bps, bid_oas_bps, ask_oas_bps, bid_ask_bps.
    """
    if tickers is None:
        return spreads.copy()
    return spreads[spreads["ticker"].isin(tickers)].copy()


def fundamental_trend():
    """
    Returns the change in key metrics from first to last quarter for each company.
    Columns: ticker, company_name, lev_start, lev_end, lev_chg,
    cov_start, cov_end, cov_chg, margin_start, margin_end, margin_chg,
    plus bond reference fields.
    """
    first_q = fin["quarter"].min()
    last_q = fin["quarter"].max()
    start = fin[fin["quarter"] == first_q][["ticker", "leverage_ratio", "interest_coverage", "ebitda_margin"]].copy()
    start.columns = ["ticker", "lev_start", "cov_start", "margin_start"]
    end = fin[fin["quarter"] == last_q][["ticker", "leverage_ratio", "interest_coverage", "ebitda_margin"]].copy()
    end.columns = ["ticker", "lev_end", "cov_end", "margin_end"]
    t = start.merge(end, on="ticker", how="inner")
    t["lev_chg"] = t["lev_end"] - t["lev_start"]
    t["cov_chg"] = t["cov_end"] - t["cov_start"]
    t["margin_chg"] = t["margin_end"] - t["margin_start"]
    t = t.merge(bonds[["ticker", "company_name", "initial_rating", "liquidity_tier", "amount_outstanding_mm"]], on="ticker")
    return t.sort_values("lev_chg", ascending=False)


# Quick reference
print("Helper functions available:")
print("  latest_snapshot()       → one row per company: latest financials + spread + bond info")
print("  quarterly_with_spreads()→ financials with quarter-end spread, all quarters")
print("  spread_history(tickers) → daily spread data for given tickers (or all)")
print("  fundamental_trend()     → change in leverage/coverage/margin from first to last quarter")
print()
print("DataFrames loaded: fin, spreads, bonds, events")
print()

# Show a preview
snap = latest_snapshot()
print(f"latest_snapshot() → {snap.shape[0]} companies, {snap.shape[1]} columns")
print(f"  Sample columns: {list(snap.columns[:8])} + {snap.shape[1]-8} more")

---
## Step 1: Exploratory Analysis

Start your analysis here. Some questions to consider:

- What's the spread distribution across the sector? Who's wide, who's tight?
- What fundamentals are most correlated with spread?
- Which companies have improved or deteriorated over time?
- Any data quality issues worth flagging?

In [ ]:
# Your analysis here

---
## Step 2: Build a Model

*(Start this after our discussion.)*

In [ ]:
# Your model here